In [0]:
catalog = "ds_mini_project_catalog"   # your catalog name
schema = "powerbi_schema"             # your schema name

# Step 1: Create catalog if not exists
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# Step 2: Use the catalog
spark.sql(f"USE CATALOG {catalog}")

# Step 3: Create schema inside the catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

# Step 4: Use the schema
spark.sql(f"USE {schema}")

In [0]:
import pandas as pd
import os

base_path = "../data/07_powerbi"

files = os.listdir(base_path)

for file in files:
    if file.endswith(".csv"):
        file_path = os.path.join(base_path, file)
        
        print(f"Processing {file}")
        
        # Read CSV
        df = pd.read_csv(file_path)
        # Drop unnamed index columns
        df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
        # Replace invalid characters in column names
        df.columns = df.columns.str.replace(r'[ ,;{}()\n\t=]', '_', regex=True)
        # Convert Pandas → Spark
        sdf = spark.createDataFrame(df)
        
        # Table name
        table_name = file.replace(".csv", "").lower()
        
        # Write to Delta table
        sdf.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"ds_mini_project_catalog.powerbi_schema.{table_name}")